In [3]:
!pip install pandas
!pip install pandasql

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26894 sha256=07e63738d9e795cb7b7f0ba34fe96444339e2668f64c580a0f15f6bf08de59cb
  Stored in directory: c:\users\vania\appdata\local\pip\cache\wheels\b4\d0\8c\a6b366870bf041849cd96e03b71641e082f8d6456269b603b7
Successfully built pandasql


In [1]:
import pandas as pd

from pandasql import sqldf

#adding r before file path, lets python ignore the backslashes and will run!
df = pd.read_csv(r"C:\Users\vania\OneDrive\Documents\anaconda_projects\PS_20174392719_1491204439457_log.csv", nrows=10000)

#creating a quick helper to make querying easier
fraud = lambda q: sqldf(q, globals())

#WRITE SQL QUERY ('df' as table name 
query_all = """
SELECT * FROM df;
"""

result_df = fraud(query_all)
result_df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,7,PAYMENT,466.73,C517929856,881.0,414.27,M2128130537,0.0,0.0,0,0
9996,7,PAYMENT,8239.66,C1483641522,11515.0,3275.34,M1108211033,0.0,0.0,0,0
9997,7,PAYMENT,6063.34,C728110179,31409.0,25345.66,M277524255,0.0,0.0,0,0
9998,7,TRANSFER,317806.64,C1021138110,10901.0,0.00,C1935506329,103168.0,0.0,0,0


play and learn pandas

In [5]:
# use sql first to flag suspicious activity 
# then use python to do the automation
#do this on my pwn, rn learning through pandas video 15 min in 

In [2]:
#old balance - new balance and then check if differnce is masssive, transaction
# difference = oldbalanceOrg - newbalanceOrig
# if difference > = 8900
# return differnece
query_select = """
SELECT oldbalanceOrg, 
    newbalanceOrig, 
    (oldbalanceOrg - newbalanceOrig) AS difference
FROM df
WHERE (oldbalanceOrg - newbalanceOrig) >= 8900;
"""

fraud(query_select)



,oldbalanceOrg,newbalanceOrig,difference
0,170136.00,160296.36,9839.64
1,41554.00,29885.86,11668.14
2,10127.00,0.00,10127.00
3,15325.00,0.00,15325.00
4,11299.00,1996.21,9302.79
...,...,...,...
1734,29655.84,0.00,29655.84
1735,14062.31,0.00,14062.31
1736,174072.00,162561.53,11510.47
1737,12299.00,0.00,12299.00


In [3]:
#check if any accounts are emptied
#oldbalanceOrg goes all the way newbalanceOrig to 0

query_high_amounts = """
SELECT oldbalanceOrg, newbalanceOrig, amount
FROM df
WHERE oldbalanceOrg > 0
    AND newbalanceOrig = 0
    AND amount >= 9900;
"""
fraud(query_high_amounts)

,oldbalanceOrg,newbalanceOrig,amount
0,10127.00,0.0,11633.76
1,15325.00,0.0,229133.94
2,705.00,0.0,215310.30
3,10835.00,0.0,311685.89
4,26845.41,0.0,110414.71
...,...,...,...
1095,29655.84,0.0,118577.00
1096,14062.31,0.0,19080.06
1097,1042.00,0.0,246846.74
1098,12299.00,0.0,564581.34


In [4]:
#check that money didnt just vanish, exp account a tranfers 50000, but only 10000 appear
#so expected balance - actual balance 
#expected = oldbalanceOrg - amoint and missing amount:oldbalanceOrg - amoint - newbalanceOrig
transfer_types_query = """
SELECT 
    type, 
    nameOrig,
    amount, 
    oldbalanceOrg, 
    newbalanceOrig, 
    (oldbalanceOrg - amount ) AS expectedNewBalance, 
    (oldbalanceOrg - amount - newbalanceOrig) AS missingAmount
FROM df 
WHERE type IN ("TRANSFER", "CASH_OUT")
    AND newbalanceOrig != (oldbalanceOrg - amount);
"""

fraud(transfer_types_query)

,type,nameOrig,amount,oldbalanceOrg,newbalanceOrig,expectedNewBalance,missingAmount
0,CASH_OUT,C905080434,229133.94,15325.00,0.0,-213808.94,-213808.94
1,TRANSFER,C1670993182,215310.30,705.00,0.0,-214605.30,-214605.30
2,TRANSFER,C1984094095,311685.89,10835.00,0.0,-300850.89,-300850.89
3,CASH_OUT,C768216420,110414.71,26845.41,0.0,-83569.30,-83569.30
4,CASH_OUT,C1570470538,56953.90,1942.02,0.0,-55011.88,-55011.88
...,...,...,...,...,...,...,...
2073,TRANSFER,C968373819,765639.59,138036.00,0.0,-627603.59,-627603.59
2074,CASH_OUT,C355447775,118577.00,29655.84,0.0,-88921.16,-88921.16
2075,TRANSFER,C47418566,246846.74,1042.00,0.0,-245804.74,-245804.74
2076,TRANSFER,C2060124480,564581.34,12299.00,0.0,-552282.34,-552282.34


In [12]:
multiple_transaction = """
SELECT 
    nameOrig,
    COUNT(*) AS total_transactions,
    SUM(amount) AS total_amount_spent,
    AVG(amount) AS average_transaction_size
FROM df
WHERE type IN ("TRANSFER", "CASH_OUT")
    AND amount < 10000 -- looking for small transaction sizes
GROUP BY nameOrig
HAVING COUNT(*) > 2 -- flags accounts making more than 2 small transactions
ORDER BY total_transactions DESC; -- flags account when more than 2 small transactions 
"""

fraud(multiple_transaction)

,nameOrig,total_transactions,total_amount_spent,average_transaction_size


In [15]:
#Check how many of your flagged transactions were actually labeled as fraud
flagged_transactions = """
SELECT 
    isFraud, 
    COUNT(*) AS count
FROM df
WHERE type IN ('TRANSFER', 'CASH_OUT')
  AND newbalanceOrig != (oldbalanceOrg - amount)
GROUP BY isFraud;
"""

fraud(flagged_transactions)

,isFraud,count
0,0,2075
1,1,3
